<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/sample-transformer-architecture/blob/predicting_model/TransformerArchitecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [117]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import random

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super(TransformerBlock, self).__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.layer_norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.layer_norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.layer_norm2(x + ff_out)
        return x

# Simple Transformer Model
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_len=5000):
        super(SimpleTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len)
        self.transformer_blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.positional_encoding(x)
        for block in self.transformer_blocks:
            x = block(x)
        logits = self.fc_out(x)
        return logits

# Custom Tokenizer (for word-based tokenization)
class SimpleTokenizer:
    def __init__(self, vocab=None):
        if vocab is None:
            vocab = {"<PAD>": 0, "<UNK>": 1}  # Add special tokens
        self.vocab = vocab
        self.inv_vocab = {v: k for k, v in vocab.items()}

    def build_vocab(self, corpus):
        words = set(corpus.split())
        for word in words:
            if word not in self.vocab:
                self.vocab[word] = len(self.vocab)
        self.inv_vocab = {v: k for k, v in self.vocab.items()}

    def encode(self, text):
        return [self.vocab.get(word, self.vocab["<UNK>"]) for word in text.split()]

    def decode(self, tokens):
        return " ".join([self.inv_vocab.get(token, "<UNK>") for token in tokens])

# Prepare Dataset
class TextDataset(Dataset):
    def __init__(self, corpus, tokenizer, seq_length=3):
        self.tokenizer = tokenizer
        self.seq_length = seq_length
        self.text = corpus.split('\n')  # Split into sentences
        self.data = self.create_data()

    def create_data(self):
        data = []
        for sentence in self.text:
            tokens = self.tokenizer.encode(sentence)
            for i in range(len(tokens) - self.seq_length):
                x = tokens[i:i+self.seq_length]
                y = tokens[i+1:i+self.seq_length+1]  # Predict the next word
                data.append((x, y))
        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, y = self.data[idx]
        return torch.tensor(x), torch.tensor(y)

# Load data and build vocabulary
corpus = """
You can ask the flowers.
I sit for hours.
Telling all the blue birds.
The bill and coo birds.
Pretty Little baby
"""
tokenizer = SimpleTokenizer()
tokenizer.build_vocab(corpus)

# Dataset and DataLoader
dataset = TextDataset(corpus, tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# Model and optimizer
vocab_size = len(tokenizer.vocab)
d_model = 512
n_heads = 8
n_layers = 6
d_ff = 2048
max_len = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleTransformer(vocab_size, d_model, n_heads, n_layers, d_ff).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(10):
    model.train()
    for x_batch, y_batch in dataloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits.view(-1, vocab_size), y_batch.view(-1))
        loss.backward()
        optimizer.step()

# Sample Prediction with Sampling (Instead of greedy)
def predict_next_word(sentence, model, tokenizer):
    model.eval()
    tokens = tokenizer.encode(sentence)  # Tokenize the sentence
    input_tensor = torch.tensor(tokens).unsqueeze(0).to(device)  # Add batch dimension
    with torch.no_grad():
        logits = model(input_tensor)  # Get logits for the next word
    # Softmax to get probabilities
    probabilities = torch.softmax(logits[:, -1, :], dim=-1)
    # Sample from the distribution (instead of taking argmax)
    next_word_id = torch.multinomial(probabilities, 1).item()
    next_word = tokenizer.decode([next_word_id])
    return next_word

# Example prediction
sentence = "Preety the"
predicted_word = predict_next_word(sentence, model, tokenizer)
print(f"Predicted next word: {predicted_word}")

Predicted next word: flowers.
